In [1]:
import pandas as pd
from statsmodels.tsa.ardl import ardl_select_order

df = pd.read_csv(
    "../data/processed/passthrough_dataset.csv",
    parse_dates=["date"]
).set_index("date")

# policy dummy: diesel subsidy removal (June 2024)
df["diesel_policy"] = (df.index >= "2024-06-01").astype(int)

df = df[["diesel", "wti", "usdmyr", "diesel_policy"]]
df = df.asfreq("ME")

df.head()


,diesel,wti,usdmyr,diesel_policy
date,,,,
2020-01-31,2.1800,57.519048,4.080110,0
2020-02-29,2.1340,50.542632,4.159900,0
2020-03-31,1.8150,29.207727,4.298409,0
2020-04-30,1.4675,16.547619,4.352486,0
2020-05-31,1.4740,28.562500,4.340906,0


In [2]:
sel = ardl_select_order(
    endog=df["diesel"],
    exog=df[["wti", "usdmyr", "diesel_policy"]],
    maxlag=4,
    maxorder=4,
    ic="aic",
    trend="c"
)

print(sel.model)


In [3]:
ardl_d_wti = sel.model.fit()
print(ardl_d_wti.summary())


                              ARDL Model Results                              
Dep. Variable:                 diesel   No. Observations:                   71
Model:                     ARDL(4, 1)   Log Likelihood                 117.350
Method:               Conditional MLE   S.D. of innovations              0.042
Date:                Thu, 08 Jan 2026   AIC                           -218.699
Time:                        09:28:31   BIC                           -201.062
Sample:                    05-31-2020   HQIC                          -211.720
                         - 11-30-2025                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const                0.2837      0.081      3.511      0.001       0.122       0.445
diesel.L1            1.1577      0.053     21.794      0.000       1.051       1.264
diesel.L2           -0.4209 

In [4]:
import numpy as np
import statsmodels.api as sm
from scipy import stats
from statsmodels.tsa.tsatools import lagmat
from statsmodels.stats.diagnostic import het_breuschpagan
res = ardl_d_wti

In [5]:
def bg_test_ardl_aligned(res, nlags=2):
    X0_full = np.asarray(res.model.data.exog)
    u = np.asarray(res.resid)

    n_u = len(u)
    if n_u <= nlags + 5:
        raise ValueError(f"Too few residual observations ({n_u}) for nlags={nlags}.")

    # align exog to residual sample
    X0 = X0_full[-n_u:, :]

    # auxiliary regression
    U_lags = lagmat(u, maxlag=nlags, trim="both")
    y = u[nlags:]
    X = np.column_stack([X0[nlags:, :], U_lags])

    aux_res = sm.OLS(y, X).fit()

    n = aux_res.nobs
    lm = n * aux_res.rsquared
    lm_p = stats.chi2.sf(lm, nlags)

    k = X.shape[1]
    R = np.zeros((nlags, k))
    R[:, -nlags:] = np.eye(nlags)
    f_test = aux_res.f_test(R)

    return {
        "LM stat": float(lm),
        "LM p-value": float(lm_p),
        "F stat": float(f_test.fvalue),
        "F p-value": float(f_test.pvalue),
        "nobs_aux": int(n),
        "resid_len": int(n_u),
        "exog_rows_full": int(X0_full.shape[0]),
    }

bg_out = bg_test_ardl_aligned(res, nlags=2)
bg_out


{'LM stat': 2.724206973894306,
 'LM p-value': 0.2561214617794595,
 'F stat': 0.6641371553177181,
 'F p-value': 0.5184599079055661,
 'nobs_aux': 65,
 'resid_len': 67,
 'exog_rows_full': 71}

In [6]:
def bp_test_ardl(res):
    u = np.asarray(res.resid)
    X0_full = np.asarray(res.model.data.exog)
    X0 = X0_full[-len(u):, :]
    X0 = sm.add_constant(X0, has_constant="add")

    bp = het_breuschpagan(u, X0)

    return {
        "LM stat": float(bp[0]),
        "LM p-value": float(bp[1]),
        "F stat": float(bp[2]),
        "F p-value": float(bp[3]),
        "k_exog": int(X0.shape[1]),
    }

bp_out = bp_test_ardl(res)
bp_out


{'LM stat': 19.118927385628037,
 'LM p-value': 0.00025834688041840014,
 'F stat': 8.385306618583884,
 'F p-value': 8.997018618123049e-05,
 'k_exog': 4}

In [7]:
import joblib
joblib.dump(res, "../data/joblib/ardl_diesel_wti.joblib")

['../data/joblib/ardl_diesel_wti.joblib']